# FlightInsight — Data Exploration

Live flight data from OpenSky Network. No server needed — this notebook calls the API directly.

**Sections:**
1. Fetch live data (Europe bbox)
2. Basic stats
3. Top countries
4. Altitude distribution
5. Speed distribution
6. Live map
7. Eval questions

In [ ]:
import os
import httpx
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

OPENSKY_CLIENT_ID = os.getenv("OPENSKY_CLIENT_ID", "")
OPENSKY_CLIENT_SECRET = os.getenv("OPENSKY_CLIENT_SECRET", "")

# Europe bounding box
BBOX_EUROPE = dict(lamin=35.0, lomin=-10.0, lamax=70.0, lomax=40.0)
BBOX_WORLD  = dict(lamin=-90, lomin=-180, lamax=90, lomax=180)

## 1. Fetch live data

In [ ]:
def get_token() -> str | None:
    if not OPENSKY_CLIENT_ID:
        return None
    r = httpx.post(
        "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token",
        data={
            "grant_type": "client_credentials",
            "client_id": OPENSKY_CLIENT_ID,
            "client_secret": OPENSKY_CLIENT_SECRET,
        },
        timeout=10,
    )
    r.raise_for_status()
    return r.json()["access_token"]


def fetch_states(bbox: dict) -> list[list]:
    token = get_token()
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    r = httpx.get(
        "https://opensky-network.org/api/states/all",
        params=bbox,
        headers=headers,
        timeout=15,
    )
    r.raise_for_status()
    return r.json().get("states") or []


COLUMNS = [
    "icao24", "callsign", "origin_country", "time_position",
    "last_contact", "longitude", "latitude", "baro_altitude",
    "on_ground", "velocity", "true_track", "vertical_rate",
    "sensors", "geo_altitude", "squawk", "spi", "position_source",
]

print("Fetching Europe live flights...")
states = fetch_states(BBOX_EUROPE)
df = pd.DataFrame(states, columns=COLUMNS[:len(states[0])] if states else COLUMNS)
df["callsign"] = df["callsign"].str.strip()
print(f"Got {len(df)} flights")
df.head()

## 2. Basic stats

In [ ]:
airborne = df[df["on_ground"] == False]
on_ground = df[df["on_ground"] == True]

print(f"Total flights tracked  : {len(df)}")
print(f"Airborne               : {len(airborne)} ({len(airborne)/len(df)*100:.1f}%)")
print(f"On ground              : {len(on_ground)}")
print(f"Countries represented  : {df['origin_country'].nunique()}")
print(f"Avg altitude (m)       : {airborne['baro_altitude'].mean():.0f}")
print(f"Avg speed (m/s)        : {airborne['velocity'].mean():.1f}  →  {airborne['velocity'].mean()*3.6:.0f} km/h")
print(f"Max speed (m/s)        : {airborne['velocity'].max():.1f}  →  {airborne['velocity'].max()*3.6:.0f} km/h")

## 3. Top countries

In [ ]:
top_countries = (
    df["origin_country"]
    .value_counts()
    .head(15)
    .reset_index()
    .rename(columns={"index": "country", "origin_country": "count"})
)

fig = px.bar(
    top_countries,
    x="origin_country",
    y="count",
    title="Top 15 countries by number of flights (Europe, live)",
    labels={"origin_country": "Country", "count": "Flights"},
    color="count",
    color_continuous_scale="Blues",
)
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

## 4. Altitude distribution

In [ ]:
alt_data = airborne["baro_altitude"].dropna()

fig = px.histogram(
    alt_data,
    nbins=50,
    title="Altitude distribution — airborne flights (meters)",
    labels={"value": "Altitude (m)", "count": "Flights"},
    color_discrete_sequence=["#1f77b4"],
)
fig.add_vline(
    x=alt_data.mean(),
    line_dash="dash",
    annotation_text=f"mean {alt_data.mean():.0f}m",
)
fig.show()

# Cruising altitude band (FL300-FL400 = 9144m-12192m)
cruising = airborne[(airborne["baro_altitude"] >= 9000) & (airborne["baro_altitude"] <= 12500)]
print(f"Flights at cruising altitude (9–12.5 km): {len(cruising)} ({len(cruising)/len(airborne)*100:.1f}% of airborne)")

## 5. Speed distribution

In [ ]:
spd = airborne["velocity"].dropna() * 3.6  # m/s → km/h

fig = px.histogram(
    spd,
    nbins=60,
    title="Speed distribution — airborne flights (km/h)",
    labels={"value": "Speed (km/h)", "count": "Flights"},
    color_discrete_sequence=["#ff7f0e"],
)
fig.add_vline(
    x=spd.mean(),
    line_dash="dash",
    annotation_text=f"mean {spd.mean():.0f} km/h",
)
fig.show()

print("Speed percentiles (km/h):")
print(spd.describe(percentiles=[0.1, 0.5, 0.9]).to_string())

## 6. Live map — all airborne flights

In [ ]:
map_df = airborne[["icao24", "callsign", "origin_country", "latitude", "longitude", "baro_altitude", "velocity"]].dropna(subset=["latitude", "longitude"])
map_df = map_df.copy()
map_df["speed_kmh"] = (map_df["velocity"] * 3.6).round(0)
map_df["altitude_km"] = (map_df["baro_altitude"] / 1000).round(2)

fig = px.scatter_map(
    map_df,
    lat="latitude",
    lon="longitude",
    color="altitude_km",
    hover_name="callsign",
    hover_data={"origin_country": True, "speed_kmh": True, "altitude_km": True, "latitude": False, "longitude": False},
    color_continuous_scale="Viridis",
    zoom=3,
    center={"lat": 50, "lon": 10},
    title="Live flights over Europe",
    labels={"altitude_km": "Altitude (km)"},
)
fig.update_traces(marker_size=5)
fig.show()
print(f"Showing {len(map_df)} airborne flights")

## 7. Vertical rate — climbing vs descending

In [ ]:
vr = airborne["vertical_rate"].dropna()

climbing   = (vr > 1).sum()
descending = (vr < -1).sum()
level      = ((vr >= -1) & (vr <= 1)).sum()

fig = px.pie(
    names=["Climbing", "Level", "Descending"],
    values=[climbing, level, descending],
    title="Flight phase distribution (airborne)",
    color_discrete_sequence=["#2ecc71", "#3498db", "#e74c3c"],
)
fig.show()
print(f"Climbing: {climbing} | Level: {level} | Descending: {descending}")

## 8. Three "waouh" eval questions

These are the questions the final FlightInsight agent must answer perfectly.
They anchor the Day 12 eval dataset.

---

**Q1 — Real-time + ranking (REALTIME)**
> *"Which 5 aircraft are currently flying the fastest over Europe, and what are their callsigns and speeds?"*

Why it's impressive: requires live data, sorting, and clean formatting. Speed numbers surprise people.

---

**Q2 — Regulation + specifics (KNOWLEDGE)**
> *"My flight from Paris to New York was cancelled 10 days before departure. Under EU regulation 261/2004, what compensation am I entitled to and what are the airline's obligations?"*

Why it's impressive: the agent must correctly identify the 14-day threshold, the €600 compensation for long-haul, and the re-routing obligation — all from the RAG corpus.

---

**Q3 — Hybrid: live data + knowledge (HYBRID)**
> *"Are there currently any flights showing unusual altitude or speed patterns over France? What could explain them?"*

Why it's impressive: combines the anomaly detection model with live data and aviation knowledge to give a reasoned, expert-sounding answer.